# QAOA: Algoritmo de Optimización Cuántica

**Objetivo:** resolver MAX-CUT en un grafo de 4 nodos con QAOA p=1.

MAX-CUT: dado un grafo G=(V,E), encontrar la partición S, V\S que maximiza el número de aristas entre los dos grupos.
QAOA mapea el problema al Hamiltoniano de costo C = Σ_(i,j)∈E (1 - Z_iZ_j)/2.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

# Grafo: 4 nodos, 4 aristas (ciclo + diagonal)
V = [0, 1, 2, 3]
E = [(0,1), (1,2), (2,3), (0,3)]  # ciclo cuadrado

print('Grafo MAX-CUT:')
print(f'  Nodos: {V}')
print(f'  Aristas: {E}')

In [ ]:
# Hamiltoniano de costo (diagonal en base Z)
def cost_hamiltonian_diagonal(n, edges):
    N = 2**n
    C = np.zeros(N)
    for i, j in edges:
        for state in range(N):
            zi = 1 - 2*((state >> i) & 1)  # +1 si bit=0, -1 si bit=1
            zj = 1 - 2*((state >> j) & 1)
            C[state] += (1 - zi*zj) / 2  # 1 si arista cortada, 0 si no
    return C

C_diag = cost_hamiltonian_diagonal(4, E)
print('Valor de corte por estado:')
for s in range(16):
    print(f'  |{format(s,"04b")}⟩: {int(C_diag[s])} aristas cortadas')

In [ ]:
# Circuito QAOA p=1
def qaoa_circuit(gamma, beta, n, edges):
    qc = QuantumCircuit(n)
    # Estado inicial: superposición uniforme
    qc.h(range(n))
    # Capa de costo: e^(-iγC)
    for i, j in edges:
        qc.rzz(2 * gamma, i, j)
    # Capa mixeadora: e^(-iβΣX)
    qc.rx(2 * beta, range(n))
    return qc

# Función de valor esperado ⟨C⟩
def expectation(params, n=4, edges=E):
    gamma, beta = params
    qc = qaoa_circuit(gamma, beta, n, edges)
    sv = Statevector.from_instruction(qc).data
    probs = np.abs(sv)**2
    return -np.dot(probs, cost_hamiltonian_diagonal(n, edges))  # negativo: minimizar

# Landscape γ,β
G, B = np.meshgrid(np.linspace(0, np.pi, 25), np.linspace(0, np.pi/2, 25))
Z_land = np.array([[-expectation([g,b]) for g in G[0]] for b in B[:,0]])

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.contourf(G, B, Z_land, levels=20, cmap='viridis')
plt.colorbar(label='⟨C⟩')
plt.xlabel('γ'); plt.ylabel('β'); plt.title('Landscape QAOA p=1')
plt.tight_layout(); plt.savefig('qaoa_landscape.png', dpi=80); plt.show()

In [ ]:
# Optimización
result = minimize(expectation, [np.pi/4, np.pi/8], method='COBYLA',
                  options={'maxiter':200})
gamma_opt, beta_opt = result.x
print(f'γ_opt = {gamma_opt:.4f}, β_opt = {beta_opt:.4f}')
print(f'⟨C⟩_opt = {-result.fun:.4f}  (máximo teórico = {len(E)})')

In [ ]:
# Ver distribución de probabilidad
qc_opt = qaoa_circuit(gamma_opt, beta_opt, 4, E)
sv = Statevector.from_instruction(qc_opt)
probs = np.abs(sv.data)**2

print('\nEstados más probables:')
idxs = np.argsort(probs)[::-1][:5]
for idx in idxs:
    label = format(idx, '04b')
    cut = int(C_diag[idx])
    print(f'  |{label}⟩: prob={probs[idx]:.3f}, corte={cut}')

## Óptimo Clásico

Para el ciclo cuadrado, el corte máximo es 4 (partición {0,2} vs {1,3}):

In [ ]:
max_cut = int(C_diag.max())
best_states = [format(s,'04b') for s in range(16) if C_diag[s] == max_cut]
print(f'Corte máximo: {max_cut}')
print(f'Estados óptimos: {best_states}')

## Ejercicio

1. Prueba QAOA con p=2 — añade una segunda capa (γ₂, β₂). ¿Mejora?
2. Cambia el grafo a un triángulo (3 nodos, 3 aristas) — ¿cuál es el MAX-CUT?
3. ¿Por qué QAOA falla para ciertos grafos con p=1?

In [ ]:
# Tu solución — triangulo
E_tri = [(0,1),(1,2),(0,2)]
C_tri = cost_hamiltonian_diagonal(3, E_tri)
print('Corte máximo (triángulo):', int(C_tri.max()))

## Próximos pasos

- **Lab completo:** `Cuadernos/laboratorios/09_qaoa_intuicion_guiada.ipynb`
- **Finance con QAOA:** `Cuadernos/laboratorios/21_optimizacion_carteras_vqe.ipynb`
- **Siguiente guía:** `10_ruido_y_decoherencia.ipynb`